In [1]:
!pip install holidays thefuzz

In [2]:
import pandas as pd
import numpy as np
import holidays

# 1. Load Data and Initial Filtering
# ---------------------------------------------------------
data_path = '../../data/tomato_dataset.csv'

df = pd.read_csv(data_path)
print(f"Raw dataset shape: {df.shape}")
print(df.head())

Raw dataset shape: (11221, 11)
         State District               Market Commodity Variety Grade  \
0  Maharashtra     Pune               Junnar    Tomato   Local   FAQ   
1  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
2  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
3  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
4  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   

  Arrival_Date  Min_Price  Max_Price  Modal_Price  Commodity_Code  
0   10/06/2021        250        750          600              78  
1   03/07/2024       2500       5500         4000              78  
2   06/07/2024       2500       6000         4000              78  
3   31/07/2024       1000       2500         2000              78  
4   01/08/2024       1000       2500         2000              78  


In [3]:
# 2. Rename columns to match onion pipeline naming convention
# ---------------------------------------------------------
df.rename(columns={
    'State': 'state', 'District': 'district', 'Market': 'mandi_name',
    'Commodity': 'commodity', 'Variety': 'variety', 'Grade': 'grade',
    'Arrival_Date': 'arrival_date', 'Min_Price': 'min_price',
    'Max_Price': 'max_price', 'Modal_Price': 'modal_price',
    'Commodity_Code': 'commodity_code'
}, inplace=True)
print(df.head())

         state district           mandi_name commodity variety grade  \
0  Maharashtra     Pune               Junnar    Tomato   Local   FAQ   
1  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
2  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
3  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   
4  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   FAQ   

  arrival_date  min_price  max_price  modal_price  commodity_code  
0   10/06/2021        250        750          600              78  
1   03/07/2024       2500       5500         4000              78  
2   06/07/2024       2500       6000         4000              78  
3   31/07/2024       1000       2500         2000              78  
4   01/08/2024       1000       2500         2000              78  


In [4]:
# 3. Filter for Pune district only
# ---------------------------------------------------------
df['district'] = df['district'].str.strip().str.title()
df = df[df['district'] == 'Pune'].copy()
print(f"After Pune filter: {len(df)} rows")
print(f"Unique districts: {df['district'].unique()}")

After Pune filter: 11221 rows
Unique districts: ['Pune']


In [5]:
# 1. Clean up basic string inconsistencies first
df['mandi_name'] = df['mandi_name'].str.strip().str.title()

# 2. How to CHECK for variants
unique_mandis = df['mandi_name'].unique()
print(f"Unique mandis BEFORE dedup ({len(unique_mandis)}):")
print(unique_mandis)

Unique mandis BEFORE dedup (18):
['Junnar' 'Junnar(Narayangaon)' 'Junnar(Narayangaon) Apmc' 'Junnar(Otur)'
 'Khed(Chakan)' 'Khed(Chakan) Apmc' 'Manchar' 'Manchar Apmc' 'Pune'
 'Pune Apmc' 'Pune(Khadiki)' 'Pune(Khadiki) Apmc' 'Pune(Manjri)'
 'Pune(Manjri) Apmc' 'Pune(Moshi)' 'Pune(Moshi) Apmc' 'Pune(Pimpri)'
 'Pune(Pimpri) Apmc']


In [6]:
from thefuzz import process

print("Potential variants to inspect:")
for mandi in unique_mandis:
    matches = process.extract(mandi, unique_mandis, limit=3)
    similar_matches = [m for m in matches if m[1] >= 85 and m[0] != mandi]
    if similar_matches:
        print(f"Original: {mandi} -> Similar to: {similar_matches}")

Potential variants to inspect:
Original: Junnar -> Similar to: [('Junnar(Narayangaon)', 90), ('Junnar(Narayangaon) Apmc', 90)]
Original: Junnar(Narayangaon) -> Similar to: [('Junnar(Narayangaon) Apmc', 95), ('Junnar', 90)]
Original: Junnar(Narayangaon) Apmc -> Similar to: [('Junnar(Narayangaon)', 95), ('Junnar', 90)]
Original: Junnar(Otur) -> Similar to: [('Junnar', 90), ('Junnar(Narayangaon)', 86)]
Original: Khed(Chakan) -> Similar to: [('Khed(Chakan) Apmc', 90)]
Original: Khed(Chakan) Apmc -> Similar to: [('Khed(Chakan)', 90), ('Pune Apmc', 86)]
Original: Manchar -> Similar to: [('Manchar Apmc', 90)]
Original: Manchar Apmc -> Similar to: [('Manchar', 90), ('Junnar(Narayangaon) Apmc', 86)]
Original: Pune -> Similar to: [('Pune Apmc', 90), ('Pune(Khadiki)', 90)]
Original: Pune Apmc -> Similar to: [('Pune', 90), ('Junnar(Narayangaon) Apmc', 86)]
Original: Pune(Khadiki) -> Similar to: [('Pune', 90), ('Pune(Khadiki) Apmc', 90)]
Original: Pune(Khadiki) Apmc -> Similar to: [('Pune', 90), ('

In [7]:
# Build a corrections dictionary to consolidate APMC variants
# Same pattern as onion pipeline
mandi_corrections = {
    'Baramati': 'Baramati', 'Indapur': 'Indapur', 'Indapur Apmc': 'Indapur',
    'Junnar': 'Junnar', 'Junnar Apmc': 'Junnar',
    'Junnar(Alephata)': 'Junnar(Alephata)', 'Junnar(Alephata) Apmc': 'Junnar(Alephata)',
    'Junnar(Narayangaon)': 'Junnar(Narayangaon)', 'Junnar(Narayangaon) Apmc': 'Junnar(Narayangaon)',
    'Junnar(Otur)': 'Junnar(Otur)', 'Junnar(Otur) Apmc': 'Junnar(Otur)',
    'Khed(Chakan)': 'Khed(Chakan)', 'Khed(Chakan) Apmc': 'Khed(Chakan)',
    'Manchar': 'Manchar', 'Manchar Apmc': 'Manchar', 'Nira': 'Nira',
    'Pune': 'Pune', 'Pune Apmc': 'Pune',
    'Pune(Khadiki)': 'Pune(Khadiki)', 'Pune(Khadiki) Apmc': 'Pune(Khadiki)',
    'Pune(Manjri)': 'Pune(Manjri)', 'Pune(Manjri) Apmc': 'Pune(Manjri)',
    'Pune(Moshi)': 'Pune(Moshi)', 'Pune(Moshi) Apmc': 'Pune(Moshi)',
    'Pune(Pimpri)': 'Pune(Pimpri)', 'Pune(Pimpri) Apmc': 'Pune(Pimpri)',
    'Shirur': 'Shirur'
}
df['mandi_name'] = df['mandi_name'].replace(mandi_corrections)

In [8]:
unique_mandis = df['mandi_name'].unique()
print(f"Unique mandis AFTER dedup ({len(unique_mandis)}):")
print(unique_mandis)

Unique mandis AFTER dedup (10):
['Junnar' 'Junnar(Narayangaon)' 'Junnar(Otur)' 'Khed(Chakan)' 'Manchar'
 'Pune' 'Pune(Khadiki)' 'Pune(Manjri)' 'Pune(Moshi)' 'Pune(Pimpri)']


In [9]:
df['variety'] = df['variety'].str.strip().str.title()
unique_varieties = df['variety'].unique()
print(f"Unique varieties: {unique_varieties}")

Unique varieties: ['Local' 'Other']


In [10]:
df['grade'] = df['grade'].str.strip().str.title()
print(df['grade'].unique())

['Faq' 'Local']


In [11]:
# Convert date to datetime object (Tomato uses DD/MM/YYYY)
df['arrival_date'] = pd.to_datetime(df['arrival_date'], dayfirst=True)

In [12]:
# No commodity filter needed - entire file is Tomato
df = df.drop(columns=['commodity_code', 'grade'], errors='ignore')

In [13]:
print(df.head())

         state district           mandi_name commodity variety arrival_date  \
0  Maharashtra     Pune               Junnar    Tomato   Local   2021-06-10   
1  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   2024-07-03   
2  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   2024-07-06   
3  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   2024-07-31   
4  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   2024-08-01   

   min_price  max_price  modal_price  
0        250        750          600  
1       2500       5500         4000  
2       2500       6000         4000  
3       1000       2500         2000  
4       1000       2500         2000  


In [14]:
# Clean up string inconsistencies
df['mandi_name'] = df['mandi_name'].str.strip().str.title()
df['district'] = df['district'].str.strip().str.title()
df['variety'] = df['variety'].str.strip().str.title()

In [15]:
df = df.sort_values(by=['mandi_name', 'variety', 'arrival_date'])

In [16]:
print(df.head(10))

            state district           mandi_name commodity variety  \
0     Maharashtra     Pune               Junnar    Tomato   Local   
29    Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
428   Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
1110  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
429   Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
30    Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
1142  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
1111  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
1112  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   
1143  Maharashtra     Pune  Junnar(Narayangaon)    Tomato   Local   

     arrival_date  min_price  max_price  modal_price  
0      2021-06-10        250        750          600  
29     2021-02-21        500        750          600  
428    2021-02-23        750       1250         1000  
1110   2021-02-24   

In [17]:
print("Analyzing date ranges and filtering sparse data...")
group_stats = df.groupby(['mandi_name', 'variety']).agg(
    start_date=('arrival_date', 'min'),
    end_date=('arrival_date', 'max'),
    record_count=('arrival_date', 'count')
).reset_index()
group_stats['possible_days'] = (group_stats['end_date'] - group_stats['start_date']).dt.days + 1
group_stats['density'] = group_stats['record_count'] / group_stats['possible_days']
print("\n--- Mandi Diagnostics ---")
print(group_stats[['mandi_name', 'variety', 'start_date', 'record_count', 'density']].to_string())

Analyzing date ranges and filtering sparse data...

--- Mandi Diagnostics ---
             mandi_name variety start_date  record_count   density
0                Junnar   Local 2021-06-10             1  1.000000
1   Junnar(Narayangaon)   Local 2021-02-21          1454  0.796712
2   Junnar(Narayangaon)   Other 2021-09-15            11  0.008359
3          Junnar(Otur)   Local 2022-06-25            56  0.046940
4          Khed(Chakan)   Other 2021-02-24          1361  0.746162
5               Manchar   Other 2021-04-29           576  0.330465
6                  Pune   Local 2021-02-21          1445  0.791347
7         Pune(Khadiki)   Local 2021-02-21          1519  0.831418
8         Pune(Khadiki)   Other 2023-03-24             1  1.000000
9          Pune(Manjri)   Other 2021-02-24          1599  0.876645
10          Pune(Moshi)   Local 2021-02-21          1479  0.809524
11         Pune(Pimpri)   Local 2021-02-21          1719  0.940887


In [18]:
# Filter Rule (relaxed for tomato): >= 100 records AND >= 20% density
valid_groups = group_stats[(group_stats['record_count'] >= 100) & (group_stats['density'] >= 0.20)]
print(f"\nKeeping {len(valid_groups)} dense combinations out of {len(group_stats)}.")
print(valid_groups[['mandi_name', 'variety', 'record_count', 'density']])


Keeping 8 dense combinations out of 12.
             mandi_name variety  record_count   density
1   Junnar(Narayangaon)   Local          1454  0.796712
4          Khed(Chakan)   Other          1361  0.746162
5               Manchar   Other           576  0.330465
6                  Pune   Local          1445  0.791347
7         Pune(Khadiki)   Local          1519  0.831418
9          Pune(Manjri)   Other          1599  0.876645
10          Pune(Moshi)   Local          1479  0.809524
11         Pune(Pimpri)   Local          1719  0.940887


In [19]:
# =========================================================
# STEP 3: Outlier Detection & Removal (IQR Method)
# =========================================================
print("Removing price outliers...")
def remove_price_outliers(group, col='modal_price'):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    group.loc[(group[col] < lower_bound) | (group[col] > upper_bound), col] = np.nan
    return group

df = df.groupby(['mandi_name', 'variety']).apply(remove_price_outliers).reset_index(drop=True)

Removing price outliers...


C:\Users\ADVIT\AppData\Local\Temp\ipykernel_10492\2311835412.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['mandi_name', 'variety']).apply(remove_price_outliers).reset_index(drop=True)


In [20]:
# =========================================================
# STEP 3.5: Resolve Same-Day Duplicates
# =========================================================
print("Resolving multiple price entries for the same day...")
aggregation_dict = {'min_price': 'mean', 'max_price': 'mean', 'modal_price': 'mean'}
if 'commodity' in df.columns:
    aggregation_dict['commodity'] = 'first'

df = df.groupby(
    ['mandi_name', 'district', 'state', 'variety', 'arrival_date']
).agg(aggregation_dict).reset_index()
print("Duplicates resolved. Ready for reindexing.")

# =========================================================
# STEP 4: Dynamic Reindexing & Zero-Arrival Flagging
# =========================================================
print("Dynamically reindexing and flagging zero-arrival days...")
df['is_real_trade'] = 1

def fill_missing_dates_dynamic(group):
    actual_start = group['arrival_date'].min()
    actual_end = group['arrival_date'].max()
    group = group.set_index('arrival_date')
    full_date_range = pd.date_range(start=actual_start, end=actual_end, freq='D')
    group = group.reindex(full_date_range)
    group.index.name = 'arrival_date'
    group['is_real_trade'] = group['is_real_trade'].fillna(0).astype(int)
    static_cols = ['mandi_name', 'district', 'state', 'variety']
    group[static_cols] = group[static_cols].ffill().bfill()
    price_cols = ['min_price', 'max_price', 'modal_price']
    group[price_cols] = group[price_cols].ffill()
    return group.reset_index()

df_continuous = df.groupby(['mandi_name', 'district', 'state', 'variety']).apply(fill_missing_dates_dynamic).reset_index(drop=True)

Resolving multiple price entries for the same day...
Duplicates resolved. Ready for reindexing.
Dynamically reindexing and flagging zero-arrival days...


C:\Users\ADVIT\AppData\Local\Temp\ipykernel_10492\1064395553.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_continuous = df.groupby(['mandi_name', 'district', 'state', 'variety']).apply(fill_missing_dates_dynamic).reset_index(drop=True)


In [21]:
print(df.columns)

Index(['mandi_name', 'district', 'state', 'variety', 'arrival_date',
       'min_price', 'max_price', 'modal_price', 'commodity', 'is_real_trade'],
      dtype='object')


In [22]:
duplicates = df[df.duplicated(subset=['arrival_date', 'mandi_name', 'variety'], keep=False)]
print(duplicates.sort_values(by=['mandi_name', 'arrival_date']).head(10))

Empty DataFrame
Columns: [mandi_name, district, state, variety, arrival_date, min_price, max_price, modal_price, commodity, is_real_trade]
Index: []


In [23]:
# =========================================================
# STEP 5: Calendar Features & Holiday Flags
# =========================================================
print("Generating calendar and holiday features...")
ind_holidays = holidays.India(years=range(2021, 2027))

df_continuous['day_of_week'] = df_continuous['arrival_date'].dt.dayofweek
df_continuous['month'] = df_continuous['arrival_date'].dt.month
df_continuous['day_of_year'] = df_continuous['arrival_date'].dt.dayofyear
df_continuous['is_weekend'] = df_continuous['day_of_week'].isin([5, 6]).astype(int)
df_continuous['is_holiday'] = df_continuous['arrival_date'].apply(lambda x: 1 if x in ind_holidays else 0)

Generating calendar and holiday features...


In [24]:
# =========================================================
# STEP: 15-Day Horizon Feature Engineering
# =========================================================
print("Calculating 15-Day Horizon specific lags and rolling features...")

df_continuous['price_lag_15'] = df_continuous.groupby('mandi_name')['modal_price'].shift(15)
df_continuous['price_lag_16'] = df_continuous.groupby('mandi_name')['modal_price'].shift(16)
df_continuous['price_lag_17'] = df_continuous.groupby('mandi_name')['modal_price'].shift(17)
df_continuous['price_lag_30'] = df_continuous.groupby('mandi_name')['modal_price'].shift(30)

df_continuous['price_roll_mean_7'] = df_continuous.groupby('mandi_name')['price_lag_15'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
df_continuous['price_roll_std_7'] = df_continuous.groupby('mandi_name')['price_lag_15'].transform(lambda x: x.rolling(window=7, min_periods=1).std())
df_continuous['price_roll_mean_30'] = df_continuous.groupby('mandi_name')['price_lag_15'].transform(lambda x: x.rolling(window=30, min_periods=1).mean())

df_continuous['price_expanding_mean'] = df_continuous.groupby('mandi_name')['price_lag_15'].transform(lambda x: x.expanding().mean())

Calculating 15-Day Horizon specific lags and rolling features...


In [25]:
# =========================================================
# STEP 7: Fourier Terms
# =========================================================
df_continuous['sin_365_1'] = np.sin(2 * np.pi * df_continuous['day_of_year'] / 365.25)
df_continuous['cos_365_1'] = np.cos(2 * np.pi * df_continuous['day_of_year'] / 365.25)
df_continuous['sin_365_2'] = np.sin(4 * np.pi * df_continuous['day_of_year'] / 365.25)
df_continuous['cos_365_2'] = np.cos(4 * np.pi * df_continuous['day_of_year'] / 365.25)

In [26]:
# =========================================================
# STEP 8: Target Variable Creation (15-day ahead)
# IMPORTANT: Create on the full continuous dataset, not on a filtered subset
# =========================================================
df_continuous['target_price'] = df_continuous.groupby('mandi_name')['modal_price'].shift(-15)

In [27]:
# =========================================================
# STEP 9: Final Cleanup & Save
# =========================================================
print("Cleaning up edge cases and saving master dataset...")
df_final = df_continuous.dropna(subset=['target_price', 'price_roll_mean_30', 'price_lag_15']).copy()

csv_filename = '../../data/tomato_cleaned_15day.csv'
df_final.to_csv(csv_filename, index=False)

print(f"Data pipeline complete! Saved {len(df_final)} clean records to '{csv_filename}'.")
print(f"Date Range: {df_final['arrival_date'].min()} to {df_final['arrival_date'].max()}")

Cleaning up edge cases and saving master dataset...
Data pipeline complete! Saved 16763 clean records to '../../data/tomato_cleaned_15day.csv'.
Date Range: 2021-03-08 00:00:00 to 2026-02-19 00:00:00
